# FINAL QSPR PIPELINE — TWO PIPELINE VERSION (B and D)

**Pipeline B:** RDKit molecular descriptors + Morgan fingerprints  
**Pipeline D:** RDKit molecular descriptors + topological descriptors + Morgan fingerprints

This notebook follows the organization of the supplied sample code and is designed for the anticancer-drug QSPR study.

## 1. Library Import

Import the cheminformatics, machine-learning, statistical, plotting, and SHAP libraries required by the workflow.

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

SEED = 42
np.random.seed(SEED)

from rdkit import Chem
from rdkit.Chem import Descriptors, rdMolDescriptors, AllChem, SaltRemover

from sklearn.model_selection import KFold, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import VarianceThreshold
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

from xgboost import XGBRegressor
import shap

plt.style.use("seaborn-v0_8-darkgrid")
plt.rcParams["figure.figsize"] = (10, 8)
plt.rcParams["font.size"] = 12

## 2. Data Loading

Load the Excel/CSV dataset and remove duplicate records.

In [ ]:
def load_data(file):
    """Load data from Excel or CSV file and remove duplicates."""
    if file.endswith(".xlsx"):
        df = pd.read_excel(file)
    else:
        df = pd.read_csv(file)

    print(f"Loaded {len(df)} rows from {file}")
    return df.drop_duplicates().reset_index(drop=True)

## 3. SMILES Standardization

Invalid SMILES are discarded, salts are removed, and canonical SMILES are generated.

In [ ]:
remover = SaltRemover.SaltRemover()

def clean_smiles(s):
    """Clean SMILES by removing salts and returning canonical SMILES."""
    try:
        mol = Chem.MolFromSmiles(s)
        if mol is None:
            return None

        mol = remover.StripMol(mol)
        return Chem.MolToSmiles(mol)
    except Exception:
        return None

## 4. RDKit Molecular Descriptors

Calculate the RDKit descriptor set used in the supplied workflow.

In [ ]:
def rdkit_features(smiles):
    """Calculate RDKit molecular descriptors."""
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            return None

        d = {
            "MolWt": Descriptors.MolWt(mol),
            "TPSA": rdMolDescriptors.CalcTPSA(mol),
            "NumRotatableBonds": rdMolDescriptors.CalcNumRotatableBonds(mol),
            "NumHDonors": rdMolDescriptors.CalcNumHBD(mol),
            "NumHAcceptors": rdMolDescriptors.CalcNumHBA(mol),
            "FractionCsp3": Descriptors.FractionCSP3(mol),
            "BalabanJ": Descriptors.BalabanJ(mol),
            "BertzCT": Descriptors.BertzCT(mol),
            "Kappa1": Descriptors.Kappa1(mol),
            "Kappa2": Descriptors.Kappa2(mol),
            "Kappa3": Descriptors.Kappa3(mol),
            "HeavyAtomCount": rdMolDescriptors.CalcNumHeavyAtoms(mol),
        }
        return d
    except Exception:
        return None

## 5. Morgan Fingerprints

Generate 1024-bit Morgan fingerprints with radius 3.

In [ ]:
def fingerprint(smiles):
    """Generate a 1024-bit Morgan fingerprint."""
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            return np.zeros(1024, dtype=int)

        return np.array(
            AllChem.GetMorganFingerprintAsBitVect(
                mol,
                radius=3,
                nBits=1024
            )
        )
    except Exception:
        return np.zeros(1024, dtype=int)

## 6. Feature Matrix — Pipeline B

Pipeline B contains RDKit descriptors and Morgan fingerprints.

In [ ]:
def build_features_B(df):
    """Build Pipeline B: RDKit descriptors + Morgan fingerprints."""
    df = df.copy()
    df["SMILES"] = df["SMILES"].apply(clean_smiles)
    df = df.dropna(subset=["SMILES"]).reset_index(drop=True)

    rd_features = df["SMILES"].apply(rdkit_features).tolist()
    rd_df = pd.DataFrame(rd_features)

    fp = np.vstack([fingerprint(s) for s in df["SMILES"]])
    fp_df = pd.DataFrame(fp, columns=[f"FP_{i}" for i in range(1024)])

    X = pd.concat([rd_df, fp_df], axis=1)
    X = X.fillna(0)
    X.columns = X.columns.astype(str)

    print(f"Pipeline B: {X.shape[1]} features generated")
    return X

## 7. Feature Matrix — Pipeline D

Pipeline D adds the ten graph-theoretic/topological descriptors available as columns in the input dataset.

In [ ]:
def build_features_D(df):
    """Build Pipeline D: RDKit + topological descriptors + Morgan fingerprints."""
    df = df.copy()
    df["SMILES"] = df["SMILES"].apply(clean_smiles)
    df = df.dropna(subset=["SMILES"]).reset_index(drop=True)

    rd_features = df["SMILES"].apply(rdkit_features).tolist()
    rd_df = pd.DataFrame(rd_features)

    fp = np.vstack([fingerprint(s) for s in df["SMILES"]])
    fp_df = pd.DataFrame(fp, columns=[f"FP_{i}" for i in range(1024)])

    topo_cols = [
        "M1", "M2", "ABC", "R", "H",
        "Wiener", "Harary", "Mostar", "PI", "Szeged"
    ]

    existing_topo_cols = [c for c in topo_cols if c in df.columns]

    if existing_topo_cols:
        X = pd.concat(
            [rd_df, df[existing_topo_cols].reset_index(drop=True), fp_df],
            axis=1
        )
        print("Topological descriptors used:", existing_topo_cols)
    else:
        X = pd.concat([rd_df, fp_df], axis=1)
        print("Warning: topological columns were not found.")

    X = X.fillna(0)
    X.columns = X.columns.astype(str)

    print(f"Pipeline D: {X.shape[1]} features generated")
    return X

## 8. Feature Filtering

Remove low-variance descriptors and highly correlated features.

In [ ]:
def filter_features(X):
    """Remove near-constant and highly correlated features."""
    vt = VarianceThreshold(0.01)
    X_vt = vt.fit_transform(X)

    X_vt = pd.DataFrame(
        X_vt,
        columns=X.columns[vt.get_support()],
        index=X.index
    )

    corr = X_vt.corr().abs()
    upper = corr.where(
        np.triu(np.ones(corr.shape), k=1).astype(bool)
    )

    drop_cols = [
        column for column in upper.columns
        if any(upper[column] > 0.95)
    ]

    X_filtered = X_vt.drop(columns=drop_cols)

    print(
        f"Feature filtering: {X.shape[1]} -> "
        f"{X_filtered.shape[1]} features"
    )
    return X_filtered

## 9. Q² Metric

In [ ]:
def Q2(y, y_pred):
    """Calculate predictive coefficient Q²."""
    press = np.sum((y - y_pred) ** 2)
    tss = np.sum((y - np.mean(y)) ** 2)

    if tss == 0:
        return 0.0

    return 1.0 - press / tss

## 10. SHAP Feature Selection

Use a Random Forest baseline to select the 30 most important features according to mean absolute SHAP value.

In [ ]:
def shap_select(X, y, n=30):
    """Select the top n features using mean absolute SHAP importance."""
    try:
        model = RandomForestRegressor(
            random_state=SEED,
            n_estimators=100
        )
        model.fit(X, y)

        explainer = shap.Explainer(model, X)
        shap_values = explainer(X)

        importance = np.abs(shap_values.values).mean(axis=0)
        n_select = min(n, len(importance))
        idx = np.argsort(importance)[::-1][:n_select]

        print(f"SHAP selection: {X.shape[1]} -> {n_select} features")
        return X.iloc[:, idx]

    except Exception as exc:
        print(f"SHAP selection failed: {exc}")
        return X

## 11. Machine-Learning Models and Hyperparameters

In [ ]:
models = {
    "Ridge": Ridge(),
    "RF": RandomForestRegressor(random_state=SEED),
    "XGB": XGBRegressor(random_state=SEED, verbosity=0)
}

param_grid = {
    "Ridge": {
        "alpha": [0.001, 0.01, 0.1, 1, 10]
    },
    "RF": {
        "n_estimators": [300, 500],
        "max_depth": [5, 10, 15]
    },
    "XGB": {
        "learning_rate": [0.01, 0.05],
        "max_depth": [3, 5, 7],
        "n_estimators": [300, 500]
    }
}

## 12. Predicted-versus-Actual Plot

In [ ]:
def plot_predicted_vs_actual(
    y_true, y_pred, property_name, pipeline_name, save_path
):
    r2 = r2_score(y_true, y_pred)
    q2 = Q2(y_true, y_pred)

    fig, ax = plt.subplots(figsize=(8, 8))

    ax.scatter(
        y_true, y_pred,
        alpha=0.6,
        edgecolors="k",
        linewidth=0.5
    )

    min_val = min(np.min(y_true), np.min(y_pred))
    max_val = max(np.max(y_true), np.max(y_pred))

    ax.plot(
        [min_val, max_val],
        [min_val, max_val],
        "r--",
        lw=2,
        label="Perfect Prediction"
    )

    z = np.polyfit(y_true, y_pred, 1)
    p = np.poly1d(z)

    order = np.argsort(y_true)
    ax.plot(
        np.asarray(y_true)[order],
        p(np.asarray(y_true)[order]),
        "g-",
        lw=1.5,
        label="Regression Line"
    )

    ax.set_xlabel("Actual Values")
    ax.set_ylabel("Predicted Values")
    ax.set_title(
        f"{property_name} - {pipeline_name}\n"
        f"Predicted vs Actual\nR² = {r2:.3f}, Q² = {q2:.3f}"
    )
    ax.legend()
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.close()

## 13. Williams Plot

In [ ]:
def calculate_leverage(X):
    """Calculate leverage values for the Williams plot."""
    from scipy.linalg import inv

    X_arr = np.asarray(X)
    X_with_intercept = np.column_stack(
        [np.ones(len(X_arr)), X_arr]
    )

    try:
        H = (
            X_with_intercept
            @ inv(X_with_intercept.T @ X_with_intercept)
            @ X_with_intercept.T
        )
        return np.diag(H)
    except Exception:
        return np.ones(len(X_arr)) * (
            X_arr.shape[1] + 1
        ) / len(X_arr)


def plot_williams_plot(
    y_true, y_pred, X, property_name, pipeline_name, save_path
):
    residuals = np.asarray(y_true) - np.asarray(y_pred)
    std = np.std(residuals)

    if std == 0:
        std_residuals = np.zeros_like(residuals)
    else:
        std_residuals = residuals / std

    leverage = calculate_leverage(X)

    n = len(X)
    p = np.asarray(X).shape[1]
    h_star = 3 * (p + 1) / n

    fig, ax = plt.subplots(figsize=(10, 6))

    ax.scatter(
        leverage,
        std_residuals,
        alpha=0.7,
        edgecolors="k",
        linewidth=0.5
    )

    ax.axhline(
        3, color="r", linestyle="--",
        linewidth=1.5, label="±3 Standardized Residuals"
    )
    ax.axhline(-
3, color="r", linestyle="--", linewidth=1.5)
    ax.axvline(
        h_star, color="b", linestyle="--",
        linewidth=1.5,
        label=f"Critical Leverage (h* = {h_star:.3f})"
    )

    outliers = (
        (np.abs(std_residuals) > 3)
        | (leverage > h_star)
    )

    if np.any(outliers):
        ax.scatter(
            leverage[outliers],
            std_residuals[outliers],
            color="red",
            s=100,
            marker="s",
            label="Outliers",
            alpha=0.7
        )

    ax.set_xlabel("Leverage (h)")
    ax.set_ylabel("Standardized Residuals")
    ax.set_title(
        f"{property_name} - {pipeline_name}\n"
        "Williams Plot (Applicability Domain)"
    )
    ax.legend()
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.close()

## 14. Distance-Based Applicability Plot

In [ ]:
def plot_distance_applicability(
    y_true, y_pred, X, property_name, pipeline_name, save_path
):
    from scipy.spatial.distance import cdist

    X_arr = np.asarray(X)

    centroid = X_arr.mean(axis=0).reshape(1, -1)
    distances = cdist(
        X_arr, centroid, metric="euclidean"
    ).ravel()

    max_distance = np.max(distances)
    distances_normalized = (
        distances / max_distance
        if max_distance > 0 else distances
    )

    residuals = np.abs(
        np.asarray(y_true) - np.asarray(y_pred)
    )
    max_residual = np.max(residuals)
    residuals_normalized = (
        residuals / max_residual
        if max_residual > 0 else residuals
    )

    fig, ax = plt.subplots(figsize=(10, 6))

    scatter = ax.scatter(
        distances_normalized,
        residuals_normalized,
        c=residuals,
        cmap="viridis",
        alpha=0.7,
        edgecolors="k",
        linewidth=0.5,
        s=80
    )

    distance_threshold = np.percentile(
        distances_normalized, 95
    )
    residual_threshold = np.percentile(
        residuals_normalized, 95
    )

    ax.axvline(
        distance_threshold,
        color="r",
        linestyle="--",
        linewidth=1.5
    )
    ax.axhline(
        residual_threshold,
        color="b",
        linestyle="--",
        linewidth=1.5
    )

    ax.set_xlabel("Normalized Distance to Model Centroid")
    ax.set_ylabel("Normalized Absolute Residual")
    ax.set_title(
        f"{property_name} - {pipeline_name}\n"
        "Distance to Model Applicability"
    )
    ax.grid(True, alpha=0.3)

    cbar = plt.colorbar(scatter)
    cbar.set_label("Absolute Residual")

    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.close()

## 15. Pipeline Runner

Run model evaluation for one target and one feature pipeline using five-fold outer CV and three-fold inner grid search.

In [ ]:
def run_single_pipeline(df, X, target, pipeline_name, out_dir):
    y = df[target].copy()

    Xt = X.copy()

    # Avoid direct target/descriptor duplication
    cols_to_remove = [
        target,
        "MolWt",
        "TPSA",
        "NumRotatableBonds",
        "SMILES"
    ]

    for col in cols_to_remove:
        if col in Xt.columns:
            Xt = Xt.drop(columns=col)

    # Feature filtering and SHAP selection
    Xt = filter_features(Xt)
    Xt = shap_select(Xt, y, n=30)

    cv_outer = KFold(
        n_splits=5,
        shuffle=True,
        random_state=SEED
    )

    results = []

    best_q2 = -np.inf
    best_summary = None

    for name, model in models.items():
        print(f"Testing {pipeline_name} | {target} | {name}")

        predictions = []
        observations = []

        for train_idx, test_idx in cv_outer.split(Xt):
            Xtrain = Xt.iloc[train_idx]
            Xtest = Xt.iloc[test_idx]
            ytrain = y.iloc[train_idx]
            ytest = y.iloc[test_idx]

            scaler = StandardScaler()
            Xtrain_scaled = scaler.fit_transform(Xtrain)
            Xtest_scaled = scaler.transform(Xtest)

            grid = GridSearchCV(
                model,
                param_grid[name],
                cv=3,
                scoring="r2",
                n_jobs=-1
            )

            # Hyperparameter selection
            grid.fit(Xtrain_scaled, ytrain)
            best_estimator = grid.best_estimator_

            best_estimator.fit(Xtrain_scaled, ytrain)
            pred = best_estimator.predict(Xtest_scaled)

            predictions.extend(pred)
            observations.extend(ytest)

        predictions = np.asarray(predictions)
        observations = np.asarray(observations)

        r2_cv = r2_score(observations, predictions)
        q2 = Q2(observations, predictions)
        rmse = np.sqrt(
            mean_squared_error(observations, predictions)
        )
        mae = mean_absolute_error(
            observations, predictions
        )

        # Refit the selected model on the full filtered feature matrix
        scaler_final = StandardScaler()
        X_full_scaled = scaler_final.fit_transform(Xt)

        final_grid = GridSearchCV(
            model,
            param_grid[name],
            cv=3,
            scoring="r2",
            n_jobs=-1
        )
        final_grid.fit(X_full_scaled, y)

        final_model = final_grid.best_estimator_
        final_model.fit(X_full_scaled, y)

        train_pred = final_model.predict(X_full_scaled)
        r2_train = r2_score(y, train_pred)

        results.append({
            "Property": target,
            "Pipeline": pipeline_name,
            "Model": name,
            "R2_train": r2_train,
            "R2_CV": r2_cv,
            "Q2": q2,
            "RMSE": rmse,
            "MAE": mae,
            "Delta_R2": r2_train - q2,
            "n_features": Xt.shape[1]
        })

        if q2 > best_q2:
            best_q2 = q2
            best_summary = {
                "Property": target,
                "Pipeline": pipeline_name,
                "Model": name,
                "R2_train": r2_train,
                "Q2": q2,
                "RMSE": rmse,
                "MAE": mae,
                "Delta_R2": r2_train - q2,
                "n_features": Xt.shape[1],
                "y_true": observations,
                "y_pred": predictions,
                "X": Xt,
                "model": final_model,
                "scaler": scaler_final
            }

    return results, best_summary

## 16. Main Pipeline

Run both Pipeline B and Pipeline D for the six target properties and save summary tables and diagnostic plots.

In [ ]:
def run_pipeline(input_file):
    base_dir = os.path.dirname(os.path.abspath(input_file))
    output_dir = os.path.join(base_dir, "QSPR_Results")

    results_dir = os.path.join(output_dir, "Results")
    plots_dir = os.path.join(output_dir, "Plots")

    os.makedirs(results_dir, exist_ok=True)
    os.makedirs(plots_dir, exist_ok=True)

    print("=" * 80)
    print("QSPR PIPELINE — PIPELINE B AND PIPELINE D")
    print("=" * 80)

    df = load_data(input_file)

    targets = [
        "logP",
        "RB",
        "TPSA",
        "C",
        "Heavy atom",
        "nPotency"
    ]
    targets = [t for t in targets if t in df.columns]

    print("Target properties:", targets)

    # Build both feature spaces
    X_B = build_features_B(df)
    X_D = build_features_D(df)

    all_results = []
    all_best = []

    for target in targets:
        print("\n" + "=" * 60)
        print("Target:", target)
        print("=" * 60)

        results_B, best_B = run_single_pipeline(
            df, X_B, target, "Pipeline_B", output_dir
        )
        all_results.extend(results_B)
        all_best.append(best_B)

        results_D, best_D = run_single_pipeline(
            df, X_D, target, "Pipeline_D", output_dir
        )
        all_results.extend(results_D)
        all_best.append(best_D)

    all_results_df = pd.DataFrame(all_results)

    best_summary = pd.DataFrame([
        {
            k: v for k, v in b.items()
            if k not in ["y_true", "y_pred", "X", "model", "scaler"]
        }
        for b in all_best
    ])

    all_results_df.to_excel(
        os.path.join(results_dir, "All_models_results.xlsx"),
        index=False
    )

    best_summary.to_excel(
        os.path.join(results_dir, "Best_model_summary.xlsx"),
        index=False
    )

    print("\nBest models by property and pipeline:")
    print(
        best_summary[
            ["Property", "Pipeline", "Model",
             "R2_train", "Q2", "RMSE", "MAE"]
        ].round(4)
    )

    return all_results_df, best_summary, all_best

## 17. Execute the Pipeline

Set the path to the Excel dataset and run the complete analysis.

In [ ]:
# Example:
# input_file = r"D:\Phd\PhD\Papers on topological indices\Dr. Saurav Malik\100-Anti cancer drugs\Dta\dataTI.xlsx"
# all_results, best_summary, best_models = run_pipeline(input_file)

# For a dataset in the current working directory:
input_file = "dataTI.xlsx"

if os.path.exists(input_file):
    all_results, best_summary, best_models = run_pipeline(input_file)
else:
    print(f"Dataset not found: {input_file}")
    print("Update input_file with the correct dataset path.")